# Tutorial 4: A Bayesian Inversion of Tide Gauge Data

Tutorial 3 built the fingerprint as a linear operator and used its adjoint to see what a
tide gauge is sensitive to. This tutorial runs the problem in the other direction. Given a
set of readings from the GLOSS network, what can be said about the ice thickness change
that produced them?

The problem is linear and Gaussian, so it has a closed-form answer. Writing $A$ for the
forward operator taking an ice thickness change $\eta$ to the vector of $n$ tide gauge
readings, the data are

$$ d = A\eta + \varepsilon, \qquad \varepsilon \sim N(0, R), $$

and with a Gaussian prior $\eta \sim N(0, Q)$ the posterior is Gaussian with

$$ \eta \mid d \;\sim\; N\!\left(QA^{*}N^{-1}d,\;\; Q - QA^{*}N^{-1}AQ\right),
\qquad N = AQA^{*} + R. $$

Everything hinges on $N$, the covariance of the predicted data. Here it acts on
$\mathbb{R}^{n}$ with $n$ a little under three hundred, while the ice thickness field has
tens of thousands of degrees of freedom. Posing the problem in the data space rather than
the model space is therefore much the cheaper option, and it is what `pygeoinf` does when
its `data_space` formalism is selected — as it is by default.

Even so, $N$ is never formed. Each of its columns would cost a full adjoint solve and a
full forward solve of the sea level equation, so assembling all $n$ of them is far more
work than solving the system iteratively. Instead $N^{-1}$ is applied by conjugate
gradients, and the iteration count is kept small by preconditioning with the same operator
built at a much lower truncation degree, where forming it densely is cheap.

The posterior is then read in three ways: maps of its expectation against the truth;
two-point covariance functions, which say how far the constraint at one place reaches and
what it implies elsewhere; and the joint distribution of the sea level contributions of
the individual ice sheets.

In [ ]:
# import the libraries either locally or installing when on colab
import time
import numpy as np
import matplotlib.pyplot as plt

try:
    import pyslfp as sl
except ImportError:
    %pip install pyslfp --quiet
    import pyslfp as sl

import pygeoinf as inf
from cartopy import crs as ccrs

from pyslfp.linear_operators import (
    FingerPrintOperator,
    TideGaugeObservationModel,
    read_gloss_tide_gauge_data,
    ice_projection_operator,
    ocean_average_operator,
)

## 1. The forward problem

The unknown is the ice thickness change, $\eta$, and the surface load it produces is
$\rho_i \eta$. Tutorial 3 used `ice_thickness_change_to_load_operator`, which additionally
multiplies by the land mask. That is not wanted here. The geometry is carried by the prior
instead, which is confined to the grounded ice sheets, so the mask in the forward operator
would be redundant — and it is worth keeping out.

The reason is that the product of two band-limited fields is not band-limited, and
truncating it back aliases. A mask inside the forward operator multiplies fields that the
prior covariance has already multiplied by a mask, and the resulting aliasing is enough to
cost $N$ its symmetry: with `ice_thickness_change_to_load_operator` in the chain it comes
out asymmetric at the level of a few parts in a hundred, which is enough to make the
Cholesky factorisation of the surrogate fail outright. With the load written simply as
$\rho_i \eta$ the same operator is symmetric to one part in $10^{8}$, and the sharp edge in
the prior costs nothing, since nothing is ever evaluated at the grounding line.

The ingredients are therefore:

- a `FingerPrintOperator` over Sobolev spaces of order two, as point evaluation requires;
- the GLOSS tide gauge model of Tutorial 3, scaled by the ice density;
- a prior with a stationary Sobolev (Matérn-type) covariance, projected onto the grounded
  ice;
- uncorrelated Gaussian observational noise.

In [ ]:
LMAX = 128             # truncation degree of the model actually inverted
SURROGATE_LMAX = 48    # truncation degree of the preconditioner

SPACE_ORDER = 2.0      # Sobolev order of the load and response spaces
SPACE_SCALE_KM = 200.0

PRIOR_ORDER = 3.0      # prior covariance: Sobolev kernel of this order,
PRIOR_SCALE_KM = 300.0 # this correlation length,
PRIOR_STD_M = 0.5      # and this pointwise standard deviation

NOISE_STD_MM = 1.0     # observational noise on each tide gauge

names, points = read_gloss_tide_gauge_data()
print(f"{len(points)} GLOSS stations")

The same construction is needed twice, at two truncation degrees, so it is worth writing
once as a function. It returns the pieces of a `LinearForwardProblem` together with the
`LinearBayesianInversion` built on it.

In [ ]:
# Builds the forward problem and inversion at the given truncation degree.
def build_model(lmax):
    state = sl.EarthState.from_defaults(lmax=lmax)
    params = state.model.parameters

    # 1. The physics, over Sobolev spaces so that point evaluation is bounded.
    space_scale = SPACE_SCALE_KM * 1.0e3 / params.length_scale
    fingerprint = FingerPrintOperator(
        state,
        load_parameters=(SPACE_ORDER, space_scale),
        response_parameters=(SPACE_ORDER, space_scale),
    )
    ice_space = fingerprint.domain

    # 2. The forward operator: load = ice density x thickness change, then the
    #    tide gauge model of Tutorial 3. No spatial mask appears here.
    gauges = TideGaugeObservationModel(fingerprint, points, names=names)
    forward_operator = gauges.forward_operator * params.ice_density

    # 3. The prior: a stationary field restricted to the grounded ice sheets.
    prior = ice_space.point_value_scaled_sobolev_kernel_gaussian_measure(
        PRIOR_ORDER,
        PRIOR_SCALE_KM * 1.0e3 / params.length_scale,
        std=PRIOR_STD_M / params.length_scale,
    ).affine_mapping(
        operator=ice_projection_operator(state, ice_space, exclude_ice_shelves=True)
    )

    # 4. The noise, and the resulting forward problem and inversion.
    noise = inf.GaussianMeasure.from_standard_deviation(
        inf.EuclideanSpace(len(points)), NOISE_STD_MM * 1.0e-3 / params.length_scale
    )
    problem = inf.LinearForwardProblem(forward_operator, data_error_measure=noise)

    return {
        "state": state,
        "params": params,
        "fingerprint": fingerprint,
        "ice_space": ice_space,
        "prior": prior,
        "problem": problem,
        "inversion": inf.LinearBayesianInversion(problem, prior),
    }

In [ ]:
exact = build_model(LMAX)

state = exact["state"]
params = exact["params"]
ice_space = exact["ice_space"]
prior = exact["prior"]
problem = exact["problem"]

# Conversions out of the non-dimensional units.
metre = params.length_scale
mm = 1000.0 * params.length_scale

print(f"Model space: {ice_space.dim} dimensions")
print(f"Data space:  {problem.data_space.dim} dimensions")

## 2. A synthetic truth and its data

`synthetic_model_and_data` draws a model from the prior, applies the forward operator and
adds a draw from the noise measure. Drawing the truth from the prior is the honest test of
a Bayesian scheme: it is the one case in which the posterior is guaranteed to be
calibrated, so any failure of the truth to sit within the stated uncertainties is a failure
of the implementation rather than of the assumptions. The consequence is that the truth is
a random smooth field rather than a realistic pattern of ice loss.

In [ ]:
# Seeded here so that the truth and the data below are reproducible.
np.random.seed(4)

truth, data = problem.synthetic_model_and_data(prior)

print(f"Truth: {truth.data.min() * metre:.2f} to {truth.data.max() * metre:.2f} m of thickness change")
print(f"Data:  standard deviation {data.std() * mm:.2f} mm, against {NOISE_STD_MM} mm of noise")

fig, ax = sl.create_map_figure(figsize=(11, 5))
sl.plot_points(
    points,
    data=data * mm,
    ax=ax,
    s=25,
    symmetric=True,
    colorbar=True,
    colorbar_kwargs={"label": "Observed sea level change (mm)"},
)
plt.show()

## 3. Solving for the posterior

`LinearBayesianInversion` assembles $N = AQA^{*} + R$ as its `normal_operator`, and
`model_posterior_measure` applies $N^{-1}$ through whatever solver is passed to it. With
`CGSolver` the operator is only ever applied, never formed.

Each application of $N$ costs one adjoint and one forward solve of the sea level equation
at `LMAX`. Forming $N$ densely would cost 290 of each, which on the machine this was
written on takes about a minute and a half. Building the same operator at
`SURROGATE_LMAX`, on the other hand, is cheap enough to form densely and factorise, and its
inverse is a good enough approximation of $N^{-1}$ to serve as a preconditioner.

The data space is small enough that the Woodbury identity, which is what makes a surrogate
preconditioner worthwhile when the data run to tens of thousands, would buy nothing here.
A dense Cholesky factorisation of a 290 by 290 matrix is immediate.

`SURROGATE_LMAX` trades one cost against the other. Raising it from 32 to 48 roughly
trebles the time spent building the preconditioner and cuts the iteration count from about
thirty to about eighteen. That is a poor bargain for a single solve and a good one here,
because the same factorisation is reused by every solve that follows: the covariance
functions and the sea level contributions below each need several more.

In [ ]:
surrogate = build_model(SURROGATE_LMAX)

start = time.time()
preconditioner = inf.CholeskySolver(galerkin=True)(surrogate["inversion"].normal_operator)
print(f"Surrogate normal operator formed and factorised in {time.time() - start:.1f} s")

The effect is easiest to see by solving twice.

In [ ]:
for label, operator in [("Without", None), ("With", preconditioner)]:
    solver = inf.CGSolver(rtol=1.0e-5)
    start = time.time()
    posterior = exact["inversion"].model_posterior_measure(
        data, solver, preconditioner=operator
    )
    posterior.expectation  # forces the solve
    print(f"{label} preconditioner: {solver.iterations} iterations in {time.time() - start:.1f} s")

The posterior from the second solve is the one kept. Whether it fits the data is worth
checking, and the natural reference is the truth itself: the misfit at the truth is
chi-squared distributed on $n$ degrees of freedom, so it should come out near 290, and the
posterior mean should fit at least as well, since it is chosen to balance misfit against
the prior.

In [ ]:
posterior_mean = posterior.expectation

print(f"chi-squared at the truth:          {problem.chi_squared(truth, data):.1f}")
print(f"chi-squared at the posterior mean: {problem.chi_squared(posterior_mean, data):.1f}")
print(f"data space dimension:              {problem.data_space.dim}")
print("Posterior mean compatible with the data at the 95% level:",
      problem.chi_squared_test(0.95, posterior_mean, data))

## 4. The posterior expectation

Mapping the expectation against the truth shows what a few hundred data can do to a field
with tens of thousands of degrees of freedom.

In [ ]:
VIEWS = {
    "Antarctica": (ccrs.SouthPolarStereo(), [-180, 180, -90, -63]),
    "Greenland": (ccrs.Orthographic(-42, 72), [-75, -8, 58, 85]),
}

limit = max(abs(truth.data).max(), abs(posterior_mean.data).max()) * metre

for name, (projection, extent) in VIEWS.items():
    fig, axes = plt.subplots(
        1, 2, figsize=(11, 5.5),
        subplot_kw={"projection": projection}, layout="constrained",
    )
    for ax, (field, label) in zip(
        axes, [(truth, "Truth (m)"), (posterior_mean, "Posterior mean (m)")]
    ):
        sl.plot(field * metre, ax=ax, map_extent=extent, vmin=-limit, vmax=limit,
                colorbar_kwargs={"label": label})
    fig.suptitle(name)
    plt.show()

The long-wavelength pattern is recovered and the rest is smoothed away, which is the
expected behaviour of a badly underdetermined problem with a smoothing prior.

## 5. Two-point covariance functions

The posterior covariance is available as an operator, and for a scalar field the most
informative thing to do with it is to evaluate the two-point covariance function
$C(x_0, x)$ at a few base points. `two_point_covariance` does this in one call: it forms
the Riesz representer of point evaluation at $x_0$ — which exists because the space is
Sobolev of order greater than one — and applies the covariance to it. Each call is one
conjugate gradient solve.

Read against the prior, these say two things. How far the constraint at a point reaches,
which is the width of the central lobe; and what the data force elsewhere when the value at
$x_0$ is raised, which is the sign and position of the outer lobes.

In [ ]:
BASE_POINTS = [
    ("East Antarctica", (-78.0, 90.0), ccrs.SouthPolarStereo(), [-180, 180, -90, -63]),
    ("West Antarctica", (-78.0, -110.0), ccrs.SouthPolarStereo(), [-180, 180, -90, -63]),
    ("Greenland", (72.0, -40.0), ccrs.Orthographic(-42, 72), [-75, -8, 58, 85]),
]

start = time.time()
fig = plt.figure(figsize=(16, 10), layout="constrained")

for column, (name, point, projection, extent) in enumerate(BASE_POINTS):
    for row, (measure, stage) in enumerate([(prior, "Prior"), (posterior, "Posterior")]):
        ax = fig.add_subplot(2, 3, 3 * row + column + 1, projection=projection)
        covariance = measure.two_point_covariance(point) * metre**2
        sl.plot(covariance, ax=ax, map_extent=extent, symmetric=True,
                colorbar_kwargs={"label": f"{stage}, {name} (m$^2$)"})
        sl.plot_points([point], ax=ax, color="k", s=60, marker="*")

plt.show()
print(f"six covariance functions in {time.time() - start:.0f} s")

The prior functions are the same isotropic blob everywhere, cut off at the ice margin, with
a peak of $0.25\,\mathrm{m}^2$, which is the square of `PRIOR_STD_M`. The posterior ones
are lower, narrower and no longer isotropic, and they carry negative lobes that the prior
does not have. Those lobes are the trade-offs the data impose: raising the thickness change
at the base point forces a compensating reduction elsewhere, because the total mass and its
low-order pattern are already fixed by the network.

The reduction in the peak is the local variance reduction, and it varies. Greenland does
best, being surrounded by the densest part of the network. Both Antarctic points retain
more variance, and the East Antarctic interior is the furthest any part of the ice sheets
gets from a tide gauge.

## 6. Sea level contributions

What is normally wanted from an inversion like this is not the field but a number: how much
each ice sheet contributed to sea level. The barystatic contribution of a load is a linear
functional of it, and Tutorial 3 showed that its kernel is the constant
$-1/(\rho_w A_o)$. The contribution of a region is therefore the integral of the thickness
change over it, times $-\rho_i / (\rho_w A_o)$, and the three regional contributions can be
had from a single `l2_products_operator`.

In [ ]:
# West Antarctica here includes the Peninsula.
west, east, peninsula, greenland = state.ice_basin_groupings(scheme="macro_regions")
masks, _ = state.grouped_ice_projections(groupings=[west + peninsula, east, greenland])

SHEETS = ["West Antarctica", "East Antarctica", "Greenland"]
contribution_operator = ice_space.l2_products_operator(masks) * (
    -params.ice_density / (params.water_density * state.ocean_area) * mm
)

# The total over all grounded ice, taken through the sea level equation itself.
response_space = exact["fingerprint"].codomain
total_operator = (
    ocean_average_operator(state, response_space.subspace(0))
    @ response_space.subspace_projection(0)
    @ exact["fingerprint"]
) * params.ice_density * mm

In [ ]:
def summarise(operator, labels):
    posterior_q = posterior.affine_mapping(operator=operator).with_dense_covariance()
    prior_q = prior.affine_mapping(operator=operator).with_dense_covariance()
    truth_q = operator(truth)
    posterior_covariance = posterior_q.covariance.matrix(dense=True)
    prior_covariance = prior_q.covariance.matrix(dense=True)
    rows = [
        {
            "label": label,
            "truth": truth_q[i],
            "mean": posterior_q.expectation[i],
            "std": np.sqrt(posterior_covariance[i, i]),
            "prior_std": np.sqrt(prior_covariance[i, i]),
        }
        for i, label in enumerate(labels)
    ]
    return rows, posterior_q, prior_q


sheet_rows, sheet_posterior, sheet_prior = summarise(contribution_operator, SHEETS)
total_rows, _, _ = summarise(total_operator, ["All grounded ice"])

header = f"{'Contribution (mm)':<22}{'Truth':>9}{'Posterior':>12}{'Sigma':>9}{'Prior sigma':>13}{'Reduction':>11}"
print(header)
print("-" * len(header))
for row in sheet_rows + total_rows:
    reduction = 100.0 * (1.0 - (row["std"] / row["prior_std"]) ** 2)
    print(
        f"{row['label']:<22}{row['truth']:>9.2f}{row['mean']:>12.2f}"
        f"{row['std']:>9.2f}{row['prior_std']:>13.2f}{reduction:>10.1f}%"
    )

The three ice sheets are not the whole of the grounded ice — the basin definitions leave
out ice caps and islands — so they need not sum exactly to the last row.

The last row is the striking one. The total is determined to a few hundredths of a
millimetre, an order of magnitude better than any of the parts, even though the parts are
what it is made of. That is only possible if the errors on the parts cancel, which is to
say that they are strongly correlated. A corner plot shows the correlations directly.

In [ ]:
inf.plot_corner_distributions(
    sheet_posterior,
    prior_measure=sheet_prior,
    true_values=contribution_operator(truth),
    labels=[f"{name} (mm)" for name in SHEETS],
    title="Barystatic sea level contributions",
)
plt.show()

covariance = sheet_posterior.covariance.matrix(dense=True)
deviation = np.sqrt(np.diag(covariance))
print("Posterior correlations:")
print(np.round(covariance / np.outer(deviation, deviation), 2))

The two Antarctic contributions are strongly anti-correlated. The network can see that
Antarctica as a whole lost or gained a certain amount, but it struggles to say which end of
the continent it came from, so an overestimate of one is bought at the price of an
underestimate of the other. Greenland is largely independent of both, its fingerprint being
distinct and the stations around it dense.

This is the same statement the negative lobes of the two-point covariance functions were
making, now reduced to three numbers.

## 7. What to change

The pieces above are the ones worth varying.

The **prior** sets what is admissible before any data are seen. `PRIOR_SCALE_KM` controls
how much structure a solution may have, and it trades directly against how much the data
can resolve; `PRIOR_STD_M` sets the amplitude. Neither should be chosen to flatter the
result, and the honest check is whether the truth stays within the posterior uncertainties
as they are varied.

The **network** is the other lever. `read_gloss_tide_gauge_data` takes a `filter_func`, so
stations can be dropped by name or by position and the cost read off directly from the
two-point covariance functions and the correlations above. The kernels of Tutorial 3 say
the same thing in advance, without any data at all.

The **observation model** can be swapped out entirely. `GraceObservationModel` and
`AltimetryObservationModel` present the same interface, and
`inf.LinearForwardProblem.from_direct_sum` combines several of them into a joint inversion.
Everything downstream of the forward operator in this notebook stays as it is.